In [ ]:
import os
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import random
import torch
import itertools
from tqdm import tqdm
import functools
from PIL import Image
from collections import defaultdict
from IPython.display import display, HTML
import base64
import io
from torch.utils.data import DataLoader
import torch.nn.functional as F
from transformers import get_scheduler
from torch import optim
import torch.nn as nn
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR
import matplotlib.pyplot as plt
from models import ScoreClassifier, QualityClassifier
import psycopg
from pathlib import Path
from grid_models import NsfwClassifier as GridClassifier
from openskill.models import PlackettLuce
import heapq
from math import sqrt, erf
import time
import contextlib
import gzip
import json
import pickle
import math

In [ ]:
model = QualityClassifier(768, 0.0)
model.load_state_dict(torch.load('classifier.pt'))
model.eval()

In [ ]:
DATASET_SIZE = 4096 * 2

with psycopg.connect(dbname='postgres', user='postgres', host=str(Path.cwd().parent / "pg-socket")) as conn:
	cursor = conn.cursor()
	cursor.execute('SELECT filehash FROM images WHERE embedding IS NOT NULL')
	all_filehashes = [bytes(row[0]) for row in cursor.fetchall()]
	random_filehashes = random.sample(all_filehashes, DATASET_SIZE)

	embeddings = {}
	for filehash in tqdm(random_filehashes, desc="Loading embeddings"):
		cursor.execute('SELECT embedding FROM images WHERE filehash = %s', (filehash,))
		embedding = bytes(cursor.fetchone()[0])
		embedding = torch.frombuffer(embedding, dtype=torch.float16).to(torch.float32)
		embeddings[filehash] = embedding

In [ ]:
# OpenSkill
convergence_sigma = 1.0 # stop when avg sigma is below this value
min_delta_mu = 1e-3  # and mu barely moves
window = 200  # how many recent rounds to inspect for convergence
max_rounds = 100_000  # hard safety cap
batch_size = 512  # how many pairs to score at once
z_target = 2.0    # 95 % per-item confidence
UNCERT_FRAC = 0.4 #0.25   # examine only the noisiest 25%
REFRESH_EVERY = 20
BOTTLENECK_BOOST = 10.0

openskill_model = PlackettLuce(beta=2.0)
ratings = defaultdict(openskill_model.rating)
memoized_scores = {}
pair_counts = defaultdict(int)  # times compared
_ordered_mu = []
_order_hits = 0
_bottleneck_item = None


@contextlib.contextmanager
def tic(msg):
	t0 = time.perf_counter()
	yield
	#print(f"{msg}: {time.perf_counter()-t0:.3f}s")


def _frontier():
	by_sigma = sorted(ratings.items(), key=lambda kv: kv[1].sigma, reverse=True)
	cut = max(int(len(by_sigma) * UNCERT_FRAC), batch_size*2)
	return [filehash for filehash, _ in by_sigma[:cut]]


def best_pairs2(k: int) -> list[tuple[str, str]]:
	"""
	Select `k` pairs with the highest utility:
		utility = quality_1vs1 · (σa + σb) / (1 + #previous_matches)
	Works on the current σ-frontier only and reuses the last μ-order.
	"""
	global _ordered_mu, _order_hits, _bottleneck_item

	# ── refresh μ-order lazily ──────────────────────────
	if not _ordered_mu or _order_hits >= REFRESH_EVERY:
		_ordered_mu  = sorted(_frontier(), key=lambda p: ratings[p].mu)
		_order_hits  = 0
	_order_hits += 1

	# ── build a max-heap of neighbouring pairs ─────────
	top = []  # elements: (-utility, a, b)
	for i in range(len(_ordered_mu) - 1):
		a, b = _ordered_mu[i], _ordered_mu[i + 1]
		ra, rb = ratings[a], ratings[b]

		util = openskill_model.predict_draw([[ra], [rb]]) * (ra.sigma + rb.sigma)
		key  = (a, b) if a < b else (b, a)   # symmetric key
		util /= (1 + pair_counts[key])        # penalise repeats

		if _bottleneck_item and (a == _bottleneck_item or b == _bottleneck_item):
			util *= BOTTLENECK_BOOST

		heapq.heappush(top, (-util, a, b))

	# ── pull the best `k` pairs ─────────────────────────
	pairs = []
	while top and len(pairs) < k:
		_, a, b = heapq.heappop(top)
		pairs.append((a, b))

	# ── fall back to random if heap ran dry ─────────────
	while len(pairs) < k:
		a, b = random.sample(random_filehashes, 2)
		pairs.append((a, b))

	return pairs


def cdf(z):
	return 0.5 * (1.0 + erf(z / sqrt(2.0)))


def bucket_progress(z_thr: float) -> tuple[bool, float, float, float, str | None]:
	"""
	Return (fully_confident, pct_ok, min_z, avg_z).

	pct_ok ∈ [0,1]  — share of items already ≥ z_thr away
	min_z            — bottleneck item
	avg_z            — just for monitoring
	"""
	by_score = sorted(
		ratings.items(), key=lambda kv: kv[1].ordinal(), reverse=True
	)
	n = len(by_score)
	if n < 10:
		return False, 0.0, 0.0, 0.0, None
	
	cut_idx = [int(n * p / 10) for p in range(1, 10)]
	boundaries = []
	for idx in cut_idx:
		mu_left = by_score[idx - 1][1].mu
		mu_right = by_score[idx][1].mu
		boundaries.append((mu_left + mu_right) / 2)

	ok = 0
	min_z = float("inf")
	z_sum = 0.0
	bottleneck_item_id = None

	for item_id, r in by_score:
		mu, sigma = r.mu, r.sigma
		if sigma == 0:
			continue
		nearest   = min(boundaries, key=lambda b: abs(mu - b))
		z         = abs(mu - nearest) / sigma
		if z >= z_thr:
			ok += 1
		if z < min_z:
			min_z = z
			bottleneck_item_id = item_id
		z_sum += z

	pct_ok = ok / n
	avg_z  = z_sum / n
	fully  = pct_ok == 1.0
	return fully, pct_ok, min_z, avg_z, bottleneck_item_id



def all_items_confident(z_thr: float) -> bool:
	by_score = sorted(ratings.items(), key=lambda kv: kv[1].ordinal(), reverse=True)
	n = len(by_score)
	if n < 10:
		return False
	
	boundaries = [by_score[int(n * p / 10)][1].mu for p in range(1, 10)]

	for _, r in by_score:
		mu, sigma = r.mu, r.sigma
		nearest = min(boundaries, key=lambda b: abs(mu - b))
		z = abs(mu - nearest) / sigma
		if z < z_thr:
			return False
	return True


def best_pairs(k: int):
	"""
	Return k pairs (a,b) with the highest utility score.
	O(n log n) thanks to a heap; much faster than all-pairs O(n²).
	"""
	# Sort once by μ so “closest μ” look-ups are cheap
	by_mu = sorted(ratings.items(), key=lambda kv: kv[1].mu)
	top    = []                   # max-heap of (-utility, a_idx, b_idx)

	# Sweep through the sorted list; neighbor pairs dominate utility
	for i in range(len(by_mu)-1):
		(a, ra), (b, rb) = by_mu[i], by_mu[i+1]
		u = openskill_model.predict_draw([[ra], [rb]]) * (ra.sigma + rb.sigma)
		heapq.heappush(top, (-u, a, b))

	# Pop the k best pairs; if k > len(top) we fall back to random
	pairs = []
	while top and len(pairs) < k:
		_, a, b = heapq.heappop(top)
		pairs.append((a, b))
	while len(pairs) < k:
		a, b = random.sample(random_filehashes, 2)
		pairs.append((a, b))
	return pairs


@torch.no_grad()
def step() -> float:
	"""Play one round, return mean |Δμ| so we can monitor convergence."""
	with tic("best_pairs"):
		pairs = best_pairs2(batch_size)
	todo_pairs = [pair for pair in pairs if pair not in memoized_scores]

	# batch score inference
	if len(todo_pairs) > 0:
		with tic("Model inference"):
			emb_a = torch.stack([embeddings[a] for a, _ in todo_pairs])
			emb_b = torch.stack([embeddings[b] for _, b in todo_pairs])
			logits = model(emb_a, emb_b)
			probs = torch.softmax(logits, dim=1)[:, 0].tolist()
			for (a, b), p in zip(todo_pairs, probs):
				memoized_scores[(a, b)] = p

	# rating updates
	with tic("rating updates"):
		mu_drift = 0.0
		for a, b in pairs:
			p = memoized_scores[(a, b)]
			winner, loser = (a, b) if p > 0.5 else (b, a)
			old_mu = ratings[winner].mu
			[(ratings[winner],), (ratings[loser],)] = openskill_model.rate([[ratings[winner]], [ratings[loser]]], ranks=[0, 1])
			mu_drift += abs(ratings[winner].mu - old_mu)
			memoized_scores[(a, b)] = p
			key = (a, b) if a < b else (b, a)
			pair_counts[key] += 1
	
		return mu_drift / len(pairs)

# Initialize ratings
for filehash in random_filehashes:
	_ = ratings[filehash]

# Self-terminating training loop
mu_deltas = []
for t in itertools.count(1):
	mu_deltas.append(step())

	# Keep only the most recent `window` deltas
	if len(mu_deltas) > window:
		mu_deltas.pop(0)
	
	if t % window == 0:
		global _bottleneck_item

		avg_mu_delta = sum(mu_deltas) / len(mu_deltas)
		avg_sigma = sum(rtg.sigma for rtg in ratings.values()) / len(ratings)
		fully, pct_ok, min_z, avg_z, new_bottleneck = bucket_progress(z_target)
		_bottleneck_item = new_bottleneck

		print(f"t={t:>7}  avg |Δμ|={avg_mu_delta:.5e}  "
			  f"pct_ok={pct_ok:.2%}  min_z={min_z:.2f}  avg_z={avg_z:.2f}  "
			  f"avg σ={avg_sigma:.2f}")

		if fully and avg_mu_delta < min_delta_mu:
			print(f"✓ reached {z_target=:g} in all 10 buckets after {t:,} steps")
			break
	
	if t >= max_rounds:
		print("Safety cap reached; stopping.")
		break

leaderboard = sorted(ratings.items(), key=lambda kv: kv[1].ordinal(), reverse=True)

In [ ]:
CKPT_DIR = Path("trueskill-checkpoints")
CKPT_DIR.mkdir(exist_ok=True)

def save_state_pickle(step:int):
    ts_time = time.strftime("%Y%m%d_%H%M%S")
    fname   = CKPT_DIR / f"trueskill_{step:07d}_{ts_time}.pkl.gz"
    payload = {
        "ratings":       {k: (v.mu, v.sigma) for k,v in ratings.items()},
        "pair_counts":   pair_counts,
        "mu_deltas":     mu_deltas,
        "step":          step,
		"beta":          openskill_model.beta,
    }
    with gzip.open(fname, "wb") as fp:
        pickle.dump(payload, fp, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"✓ checkpointed at step {step:,} → {fname}")

save_state_pickle(t)

## Visualise the ELO rankings

In [ ]:
with psycopg.connect(dbname='postgres', user='postgres', host=str(Path.cwd().parent / "pg-socket")) as conn:
	cursor = conn.cursor()
	filehash_to_path = {}

	for filehash in tqdm(random_filehashes, desc="Loading paths"):
		cursor.execute('SELECT path FROM images WHERE filehash = %s', (filehash,))
		path = cursor.fetchone()[0]
		filehash_to_path[filehash] = path

In [ ]:
ELO_SCALE = 400 / math.log(10)
ELO_OFFSET = 1500
def mu_to_elo(mu: float):
	return mu * (ELO_SCALE / openskill_model.beta) + ELO_OFFSET

scores = {p: mu_to_elo(rtg.mu) for p, rtg in ratings.items()}
min_score = min(scores.values())
max_score = max(scores.values())
NUM_BINS = 10
bin_size = (max_score - min_score) / NUM_BINS

rankings = {p: min(NUM_BINS - 1, int((r - min_score) / bin_size)) for p, r in scores.items()}
rankings = list((p, r) for p, r in rankings.items())


def img_html(filehash):
	image = Image.open(filehash_to_path[filehash])
	scale = 512 / max(image.size)
	image = image.resize((int(image.width * scale), int(image.height * scale)))
	image_base64 = io.BytesIO()
	image.save(image_base64, format='WebP', quality=80)
	image_base64 = base64.b64encode(image_base64.getvalue()).decode('utf-8')
	return f'<img src="data:image/webp;base64,{image_base64}" width="512" style="margin: 5px;">'

html = ""

for bin_number in tqdm(range(NUM_BINS)):
	num_images = sum(1 for _, bin in rankings if bin == bin_number)
	images = [filehash for filehash, bin in rankings if bin == bin_number]
	images = random.sample(images, 32)

	html += f"<h1>Bin {bin_number} ({num_images} images)</h1>"
	#html += "<div style='display: flex; flex-wrap: wrap;'>"
	for filehash in images:
		html += img_html(filehash)
	#html += "</div>"
	html += "<hr>"

Path("rankings.html").write_text(html)
#display(HTML(html))